# NLP Day 1 · 실습 2. BoW와 TF-IDF로 영화 리뷰 분류하기

**연결 이론:** 이론 3 — 문서 벡터, BoW, TF-IDF, 텍스트 분류, 데이터 누수

## 1. 이번 실습에서 만들 것

영화 리뷰를 숫자 벡터로 바꾸고, **같은 분류 모델에서 BoW와 TF-IDF 표현을 비교**합니다.

- **A형 함께 구현하기:** 작은 문서–단어 행렬 확인 → BoW 변환 → Logistic Regression 학습 → Accuracy·F1 평가
- **B형 직접 완성하기:** `make_tfidf_features()`에서 `fit_transform()`과 `transform()` 연결 → 동일한 분류기로 결과 비교
- **결과 해석:** 오분류 리뷰를 살펴보고 **학습 데이터에서만 어휘집과 IDF를 학습하는 이유**를 설명합니다.

**데이터 안내:** 공식 [NSMC (Naver Sentiment Movie Corpus)](https://github.com/e9t/nsmc)의 `ratings_train.txt`에서 고정된 훈련/검증 부분집합을 준비합니다. `0=부정`, `1=긍정`입니다. 최초 실행 시 다운로드하여 `assets/nlp/nsmc_train_subset.csv`, `assets/nlp/nsmc_val_subset.csv`로 저장하며, 다음 수업에서도 같은 파일을 사용할 수 있습니다. 이미 저장된 경우 다시 만들지 않습니다.

인터넷을 사용할 수 없는 경우에는 ZIP에 포함된 **별도의 인공 연습 데이터**(`demo_*.csv`)를 사용합니다. 이 자료는 NSMC가 아니므로 실제 NSMC 성능으로 해석하지 않습니다. 원본 데이터에 대한 정보는 [NSMC 저장소](https://github.com/e9t/nsmc)를 참고하세요.

실행에 필요한 라이브러리: `pandas`, `scikit-learn` (또는 기존 과정의 동일한 Python 환경).

## 2. A형 · 함께 구현하기

### 2.1 데이터 경로와 학습·검증 자료 준비

노트북은 기존 CV 실습 파일과 **같은 프로젝트 루트**에 놓고 실행합니다. 데이터는 `assets/nlp/`에서 읽고, 실행 결과는 `outputs/nlp/`에 저장합니다.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import json
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path.cwd()
DATA_DIR = ROOT / "assets" / "nlp"
OUTPUT_DIR = ROOT / "outputs" / "nlp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError("assets/nlp 폴더를 찾을 수 없습니다. ZIP에 포함된 assets/nlp 폴더를 프로젝트 루트에 풀어주세요.")

NSMC_TRAIN = DATA_DIR / "nsmc_train_subset.csv"
NSMC_VAL = DATA_DIR / "nsmc_val_subset.csv"
DEMO_TRAIN = DATA_DIR / "demo_train.csv"
DEMO_VAL = DATA_DIR / "demo_val.csv"
NSMC_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"


def prepare_data():
    # 이미 만들어진 NSMC 부분집합은 다시 샘플링하지 않습니다.
    if NSMC_TRAIN.exists() and NSMC_VAL.exists():
        return pd.read_csv(NSMC_TRAIN), pd.read_csv(NSMC_VAL), "NSMC 부분집합"

    try:
        print("NSMC 원본에서 고정된 부분집합을 준비합니다...")
        with urlopen(NSMC_URL, timeout=10) as response:
            original = pd.read_csv(BytesIO(response.read()), sep="\t")
        original = original.dropna(subset=["document", "label"]).copy()
        original["document"] = original["document"].astype(str).str.strip()
        original = original[original["document"].ne("")]
        original = original.drop_duplicates(subset=["document"])
        # 부정/긍정 각 1,250건씩 선택한 뒤 80:20 분할 (2,000/500건)
        selected = pd.concat([
            original[original["label"] == label].sample(n=1250, random_state=2026)
            for label in (0, 1)
        ], ignore_index=True)
        train_df, val_df = train_test_split(
            selected, test_size=0.20, random_state=42, stratify=selected["label"]
        )
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)
        train_df.to_csv(NSMC_TRAIN, index=False, encoding="utf-8-sig")
        val_df.to_csv(NSMC_VAL, index=False, encoding="utf-8-sig")
        print("NSMC 파일 저장 완료:", NSMC_TRAIN.name, NSMC_VAL.name)
        return train_df, val_df, "NSMC 부분집합"
    except Exception as exc:
        print("NSMC 다운로드에 실패하여 오프라인 인공 연습 데이터를 사용합니다.")
        print("오류 유형:", type(exc).__name__)
        if not (DEMO_TRAIN.exists() and DEMO_VAL.exists()):
            raise FileNotFoundError("오프라인 연습용 CSV를 찾을 수 없습니다. assets/nlp 폴더를 확인하세요.") from exc
        return pd.read_csv(DEMO_TRAIN), pd.read_csv(DEMO_VAL), "오프라인 인공 연습 데이터 (NSMC 아님)"

train_df, val_df, data_source = prepare_data()
for df in (train_df, val_df):
    df.dropna(subset=["document", "label"], inplace=True)
    df["document"] = df["document"].astype(str)
    df["label"] = df["label"].astype(int)

assert set(train_df["label"]) == {0, 1}
assert set(val_df["label"]) == {0, 1}
assert set(train_df["id"]).isdisjoint(set(val_df["id"]))
print("자료 유형:", data_source)
print("학습/검증 건수:", len(train_df), "/", len(val_df))
print("학습 라벨 분포:", train_df["label"].value_counts().sort_index().to_dict())
print("검증 라벨 분포:", val_df["label"].value_counts().sort_index().to_dict())

### 2.2 짧은 리뷰로 BoW의 행과 열 확인하기

실습의 첫 단계에서는 문서가 행, 어휘가 열인 **문서–단어 행렬**을 직접 확인합니다. 문장을 나누는 방법은 복잡한 형태소 분석 대신 간단한 공백 기준을 사용합니다.

In [ ]:
def normalize_text(text):
    # 구두점을 공백으로 바꾸고, 공백을 정리합니다.
    text = re.sub(r"[^\w\s]", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

sample_reviews = [
    "정말 재미있다.",
    "연기가 좋고 재미있다.",
    "내용이 지루하다.",
    "재미있을 줄 알았는데 지루하다.",
]

sample_vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
sample_matrix = sample_vectorizer.fit_transform([normalize_text(x) for x in sample_reviews])

sample_dtm = pd.DataFrame(
    sample_matrix.toarray(),
    index=[f"리뷰 {i}" for i in range(1, len(sample_reviews) + 1)],
    columns=sample_vectorizer.get_feature_names_out(),
)
print("행렬 Shape:", sample_matrix.shape)
print("행 = 리뷰, 열 = 어휘, 값 = 등장 횟수")
display(sample_dtm)

### 2.3 NSMC 자료를 BoW로 변환하고 분류하기

**중요:** 어휘집은 **학습 리뷰로만** 만들고 검증 리뷰에는 같은 변환기를 적용합니다. 학습/검증 데이터에서 각각 `fit_transform()`을 실행하지 않습니다.

In [ ]:
# 동일한 정규화를 학습·검증 데이터 모두에 적용합니다.
train_df["clean_text"] = train_df["document"].map(normalize_text)
val_df["clean_text"] = val_df["document"].map(normalize_text)

# 비어 있는 텍스트는 제거하여 벡터화 오류를 방지합니다.
train_df = train_df[train_df["clean_text"].ne("")].copy().reset_index(drop=True)
val_df = val_df[val_df["clean_text"].ne("")].copy().reset_index(drop=True)

X_train_text = train_df["clean_text"]
X_val_text = val_df["clean_text"]
y_train = train_df["label"]
y_val = val_df["label"]

bow_vectorizer = CountVectorizer(
    tokenizer=str.split, token_pattern=None, lowercase=False,
    min_df=2, max_features=8000,
)
X_train_bow = bow_vectorizer.fit_transform(X_train_text)  # 어휘집 학습 + 변환
X_val_bow = bow_vectorizer.transform(X_val_text)          # 같은 어휘집 적용

print("BoW 학습 Shape:", X_train_bow.shape)
print("BoW 검증 Shape:", X_val_bow.shape)
print("학습한 어휘 수:", len(bow_vectorizer.vocabulary_))
assert X_train_bow.shape[1] == X_val_bow.shape[1]

bow_model = LogisticRegression(max_iter=1000, random_state=42, solver="liblinear")
bow_model.fit(X_train_bow, y_train)
bow_preds = bow_model.predict(X_val_bow)

def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred, pos_label=1)),
    }

bow_scores = evaluate_predictions(y_val, bow_preds)
print("BoW 검증 결과:", {k: round(v, 4) for k, v in bow_scores.items()})

(OUTPUT_DIR / "day1_bow_metrics.json").write_text(
    json.dumps({"source": data_source, **bow_scores}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

### 2.4 분류 결과와 오분류 사례 확인하기

지표뿐 아니라 실제 리뷰를 함께 살펴보면 모델이 **어떤 표현을 잘못 판단했는지** 확인할 수 있습니다.

In [ ]:
error_mask = bow_preds != y_val.to_numpy()
error_examples = val_df.loc[error_mask, ["document", "label"]].copy()
error_examples["예측"] = bow_preds[error_mask]
print("BoW 오분류 개수:", len(error_examples))
if len(error_examples):
    display(error_examples.head(3).rename(columns={"document": "리뷰", "label": "정답"}))
else:
    print("이번 검증 자료에서 오분류가 없었습니다.")

## 3. B형 · 직접 완성하기

### 3.1 응용 문제: BoW를 TF-IDF로 바꾸기

같은 영화 리뷰와 같은 분류 모델을 사용하되, **입력 표현만 TF-IDF로 변경**합니다.

**과제:** `make_tfidf_features()`에서 다음 두 부분을 완성하세요.

- 학습 문장에 대해 `vectorizer.fit_transform(...)` 실행
- 검증 문장에 대해 **이미 학습한** `vectorizer.transform(...)` 실행

`TfidfVectorizer`의 기본 동작에는 벡터 길이를 정규화하는 과정 등이 포함되므로, 수업에서 직접 계산한 TF-IDF 숫자와 출력값이 완전히 같지는 않을 수 있습니다.

In [ ]:
def make_tfidf_features(train_texts, val_texts):
    vectorizer = TfidfVectorizer(
        tokenizer=str.split, token_pattern=None, lowercase=False,
        min_df=2, max_features=8000,
    )

    # TODO 1: train_texts로 변환기를 학습하고 학습 문서 벡터를 만드세요.
    X_train_tfidf = None

    # TODO 2: 학습된 같은 변환기로 검증 문서 벡터를 만드세요.
    X_val_tfidf = None

    return vectorizer, X_train_tfidf, X_val_tfidf


tfidf_vectorizer, X_train_tfidf, X_val_tfidf = make_tfidf_features(X_train_text, X_val_text)
print("B형 변환 상태:", "완료" if X_train_tfidf is not None and X_val_tfidf is not None else "TODO 미완성")

### 3.2 같은 분류기로 결과 비교하기

아래 셀은 B형의 `TODO`를 완료한 뒤 다시 실행하면 TF-IDF의 성능을 측정하고 비교표를 만들어 줍니다. **TF-IDF 성능이 반드시 BoW보다 높아야 하는 것은 아닙니다.**

In [ ]:
comparison = pd.DataFrame([{"표현 방식": "BoW", **bow_scores}])

if X_train_tfidf is None or X_val_tfidf is None:
    print("TODO가 아직 완성되지 않았습니다. TF-IDF 변환 코드를 작성하고 다시 실행하세요.")
else:
    assert X_train_tfidf.shape[0] == len(train_df)
    assert X_val_tfidf.shape[0] == len(val_df)
    assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1]

    tfidf_model = LogisticRegression(max_iter=1000, random_state=42, solver="liblinear")
    tfidf_model.fit(X_train_tfidf, y_train)
    tfidf_preds = tfidf_model.predict(X_val_tfidf)
    tfidf_scores = evaluate_predictions(y_val, tfidf_preds)
    comparison = pd.DataFrame([
        {"표현 방식": "BoW", **bow_scores},
        {"표현 방식": "TF-IDF", **tfidf_scores},
    ])
    print("TF-IDF 오분류 예시:")
    tfidf_errors = val_df.loc[tfidf_preds != y_val.to_numpy(), ["document", "label"]].copy()
    tfidf_errors["예측"] = tfidf_preds[tfidf_preds != y_val.to_numpy()]
    if len(tfidf_errors):
        display(tfidf_errors.head(3).rename(columns={"document": "리뷰", "label": "정답"}))
    else:
        print("이번 검증 자료에서 오분류가 없었습니다.")

    # 이후 실습에서 동일한 성능 비교를 재사용할 수 있습니다.
    comparison.to_csv(OUTPUT_DIR / "day1_bow_tfidf_comparison.csv", index=False, encoding="utf-8-sig")

print("자료 유형:", data_source)
display(comparison.round(4))

## 4. 결과 확인과 이번 실습 간단하게 요약

**확인할 결과:** BoW와 TF-IDF의 Accuracy·F1, 문서 벡터의 Shape, 잘못 분류한 영화 리뷰. B형 코드를 완성해 비교표에 두 행이 나타나는지 확인하세요.

**첫째,** BoW는 문서에 등장한 단어들의 횟수로 문서를 벡터화합니다.

**둘째,** TF-IDF는 단어 빈도와 문서 전체에서의 등장 분포를 함께 반영하여 가중치를 계산합니다.

**셋째,** 학습 데이터에서 만든 어휘집과 IDF를 **검증 데이터에 그대로 적용**해야 올바르게 평가할 수 있습니다.

**넷째,** BoW와 TF-IDF를 비교할 때는 같은 데이터와 같은 분류 모델을 사용하고, 결과의 차이를 실제 지표와 오분류 사례를 근거로 설명해야 합니다.

**다섯째,** 이러한 표현은 단어의 등장 정보를 다루지만 문맥과 단어 순서를 직접 표현하지 못합니다. 다음 실습에서는 **단어의 등장 순서**를 Bigram으로 표현해 다음 단어를 예측합니다.

> 저장된 `assets/nlp/nsmc_train_subset.csv`와 `nsmc_val_subset.csv`가 있다면, 이후 BERT 분류 실습에서도 **같은 분할**을 사용해 비교할 수 있습니다. 현재 자료 유형이 오프라인 연습 데이터라면 결과를 NSMC 실험 결과로 표기하지 않습니다.